# WRF-GS+ — one-room wireless RF experiment

**Tanisha Jain · Experiment notebook**

This notebook orchestrates data generation, training, validation, and evaluation using a separate `WRFGS_Fullroom_Package.zip`. **That package is required and is not distributed in this repository.** The notebook cannot run from this repository alone.

The algorithm implementation is [WRF-GS+](https://github.com/wenchaozheng/WRF-GSplus), with [Sionna RT](https://github.com/NVlabs/sionna-rt/tree/v1.2.1) for simulated RF data. The original wrapper package records upstream commit `e564f3184463717ae08660a8b7232c6318c73ca6`.

This workflow studies normalized directional RF images in one room with a fixed receiver and changing transmitter positions. It does not establish absolute RSSI accuracy or generalization to other rooms. Outputs have been cleared for publication; no fresh GPU run is claimed.

## 1. Check the GPU
Use a fresh Google Colab GPU runtime. The helper streams subprocess logs and stops on a failed command.

In [ ]:
import os, sys, subprocess, json, hashlib, zipfile
from pathlib import Path
def run(cmd, **kwargs):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env.update(kwargs.pop('env', {}))
    with subprocess.Popen([str(x) for x in cmd], stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1,
                          env=env, **kwargs) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        result = process.wait()
    if result:
        raise RuntimeError(f'Command failed (exit {result}). Read the error printed above before continuing.')
run(['nvidia-smi'])

## 2. Supply your experiment package
Place `WRFGS_Fullroom_Package.zip` in `MyDrive/WRFGS_fullroom/`, or upload it when prompted. It must contain the original `wrfgs_one_room/` layout, manifest, scripts, room assets, and pinned vendor sources. Hash checks detect changes relative to the manifest; they do not establish the trustworthiness of an unknown package. Only use your trusted original package. See [data and setup](../docs/SETUP_AND_DATA.md).

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
archive = Path('/content/WRFGS_Fullroom_Package.zip')
saved_package = Path('/content/drive/MyDrive/WRFGS_fullroom/WRFGS_Fullroom_Package.zip')
if saved_package.exists():
    import shutil
    shutil.copy2(saved_package, archive)
else:
    uploaded = files.upload()
    assert 'WRFGS_Fullroom_Package.zip' in uploaded, 'Upload the supplied package ZIP.'
    archive.write_bytes(uploaded['WRFGS_Fullroom_Package.zip'])
    del uploaded
destination = Path('/content').resolve()
with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        candidate = (destination / info.filename).resolve()
        assert candidate.is_relative_to(destination / 'wrfgs_one_room'), 'Unexpected archive path'
        assert ((info.external_attr >> 16) & 0o170000) != 0o120000, 'Symlinks are not accepted'
    z.extractall(destination)
ROOT = Path('/content/wrfgs_one_room')
os.chdir(ROOT)
manifest = json.loads((ROOT / 'PACKAGE_MANIFEST.json').read_text())
for name, expected in manifest['sha256'].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == expected, name
RUN_DIR = Path('/content/drive/MyDrive/WRFGS_fullroom')
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.environ['WRF_RUN_DIR'] = str(RUN_DIR)
print('Verified source commit:', manifest['wrfgsplus_commit'])
print('Persistent outputs:', RUN_DIR)

## 3. Create the isolated environment
The external package provides `setup_colab.py` for its Python 3.11 / PyTorch 2.6 / CUDA 12.4 / Sionna RT 1.2.1 environment. It compiles the RF rasterizer and requires a compatible CUDA compiler. Review that script before running a package obtained elsewhere.

In [ ]:
run([sys.executable, 'setup_colab.py'])
PY = '/content/wrfgs_env/bin/python'
print('Use this Python for every following experiment command:', PY)

## 4. Validate the math and GPU runtime
Run the package tests and runtime smoke check before generating the full dataset.

In [ ]:
run([PY, '-m', 'unittest', 'test_rf_math.py', '-v'])
run([PY, 'verify_runtime.py'])

## 5. Prepare the room
The package supplies the room geometry and material assignments. Inspect the generated room plan and materials before proceeding; these are modeling assumptions, not measured material properties.

In [ ]:
run([PY, 'generate_data.py', 'prepare'])
from IPython.display import display, Image, Markdown
display(Image(filename=str(RUN_DIR / 'room_plan.png')))
display(Markdown('**Material assignments:**'))
print((RUN_DIR / 'materials.json').read_text())

## 6. Pilot simulation
Inspect the RF images, line-of-sight check, convergence check, and loaded materials. The original configuration uses a 4×4 array at 3.5 GHz and per-image normalization. Planar-array ambiguity limits directional interpretation.

In [ ]:
run([PY, 'generate_data.py', 'pilot'])
display(Image(filename=str(RUN_DIR / 'pilot_rf_images.png')))
print('LOS check:', (RUN_DIR / 'los_check.json').read_text())
print('Convergence check:', (RUN_DIR / 'convergence.json').read_text())
print('Loaded radio materials:', (RUN_DIR / 'loaded_materials.json').read_text())

## 7. Generate the dataset
The original package uses 600 transmitter locations: 420 training, 90 validation, and 90 test samples. Generation can resume using its cache.

In [ ]:
run([PY, 'generate_data.py', 'all'])
print((RUN_DIR / 'data_ready.json').read_text())

## 8. Train with checkpoints
Train to 10,000 iterations, resuming when a checkpoint exists. Checkpoints are loaded using pickle-compatible deserialization; only load your own trusted checkpoints.

In [ ]:
checkpoint = RUN_DIR / 'model' / 'last.pt'
if checkpoint.exists():
    step = int(subprocess.check_output([PY, '-c',
        "import torch; print(torch.load(r'" + str(checkpoint) + "', map_location='cpu', weights_only=False)['iteration'])"], text=True).strip())
else:
    step = 0
if step < 10000:
    cmd = [PY, 'train_room.py', 'train', '--iterations', '10000']
    if checkpoint.exists(): cmd.append('--resume')
    run(cmd)
else:
    print('Checkpoint has already reached iteration', step)

## 9. Inspect validation performance
Review MAE and SSIM before deciding whether to run the optional continuation cell. Do not use the test split for this decision.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
history_path = RUN_DIR / 'model' / 'history.csv'
if not history_path.exists():
    raise RuntimeError(f'Training history was not created at {history_path}. Check the training cell output and WRF_RUN_DIR. Do not restart data generation or create a blank CSV.')
if history_path.stat().st_size == 0:
    raise RuntimeError('Training history is empty. Check whether training is still running or failed before its first validation.')
history = pd.read_csv(history_path)
if history.empty:
    raise RuntimeError('No validation rows yet. Wait for the first validation/checkpoint or inspect the training error.')
display(history.tail())
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history.iteration, history.val_mae)
axes[0].set(xlabel='Iteration', ylabel='Validation MAE')
axes[1].plot(history.iteration, history.val_ssim)
axes[1].set(xlabel='Iteration', ylabel='Validation SSIM')
plt.show()

In [ ]:
# OPTIONAL: change to True only if you choose to continue based on validation.
CONTINUE_TRAINING = False
if CONTINUE_TRAINING:
    run([PY, 'train_room.py', 'train', '--iterations', '20000', '--resume'])
# The same schedule supports a later extension to 30000; do not exceed it.

## 10. Final held-out evaluation
The package evaluates the fixed test split and compares against nearest-training and mean-training baselines. Metrics describe the normalized RF image task in this room. The comparison image and report are generated only after a successful run.

In [ ]:
run([PY, 'train_room.py', 'evaluate'])
summary = json.loads((RUN_DIR / 'evaluation' / 'summary.json').read_text())
print(json.dumps(summary, indent=2))
display(Image(filename=str(RUN_DIR / 'evaluation' / 'comparisons.png')))
display(Image(filename=str(RUN_DIR / 'evaluation' / 'validation_curve.png')))
display(Markdown((RUN_DIR / 'evaluation' / 'mentor_report.md').read_text()))

## 11. Archive the completed run
This saves the source package and results in your own Google Drive. The archive can be large.

In [ ]:
import shutil
assert (RUN_DIR / 'evaluation' / 'summary.json').exists(), 'Final evaluation has not completed.'
shutil.copy2('/content/WRFGS_Fullroom_Package.zip', RUN_DIR.parent / 'WRFGS_fullroom_source.zip')
result_zip = shutil.make_archive(str(RUN_DIR.parent / 'WRFGS_fullroom_completed_run'), 'zip', root_dir=RUN_DIR)
print('Saved completed run:', result_zip)
# Optional download:
# files.download(result_zip)

## Scope and attribution
This is an experiment runner around published research and the separate room-specific adapter package. It is not an independent implementation of the WRF-GS+ algorithm. No numerical WRF-GS+ score is claimed here because this source notebook contains no saved final metric outputs. See [credits](../THIRD_PARTY_NOTICES.md).